# Model Tuning & Deployment

After building a baseline model, **hyperparameter tuning** squeezes out better performance.
Then **saving the model** lets you reuse it without retraining every time.

## What's Covered

| Topic | Tool | Purpose |
|---|---|---|
| **Grid Search** | `GridSearchCV` | Try every combination of hyperparameters exhaustively |
| **Random Search** | `RandomizedSearchCV` | Sample random combinations — faster for large grids |
| **Cross-Validation** | `cross_val_score`, `StratifiedKFold` | More reliable evaluation than a single train/test split |
| **Pipeline** | `Pipeline`, `ColumnTransformer` | Chain preprocessing + model into one reusable object |
| **Model Saving** | `joblib`, `pickle` | Persist trained model to disk, reload for inference later |

## What Are Hyperparameters?

- **Model weights** (e.g. linear regression slopes) → learned automatically from data by `.fit()`
- **Hyperparameters** (e.g. `max_depth`, `n_neighbors`, `C`) → **you must set them before training**
- Choosing the right hyperparameter values can significantly improve accuracy
- Finding them manually by trial-and-error is tedious → `GridSearchCV` / `RandomizedSearchCV` automates this

In [ ]:
## ── GRIDSEARCHCV & RANDOMIZEDSEARCHCV ────────────────────────────────────────

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV,
    cross_val_score, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from scipy.stats import randint

# Synthetic classification dataset — predict student pass/fail from 5 features
X, y = make_classification(
    n_samples=500, n_features=5, n_informative=4, n_redundant=1,
    random_state=42
)
feature_names = ['StudyHours', 'Attendance', 'PastScore', 'Sleep', 'Internet']
X = pd.DataFrame(X, columns=feature_names)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print('=== Dataset ===')
print(f'Train: {len(X_train)} samples  |  Test: {len(X_test)} samples')
print(f'Class distribution — Fail: {(y==0).sum()}, Pass: {(y==1).sum()}')

# ── BASELINE: Default hyperparameters ─────────────────────────────────────────
base_model = DecisionTreeClassifier(random_state=42)  # default hyperparams
base_model.fit(X_train_s, y_train)
base_acc = accuracy_score(y_test, base_model.predict(X_test_s))
print(f'\nBaseline Decision Tree (all defaults): {base_acc:.4f} accuracy')

# ── GRID SEARCH CV ────────────────────────────────────────────────────────────
# GridSearchCV tries EVERY combination in param_grid
# cv=5 → 5-fold cross-validation for each combination (more reliable than a single split)
# scoring='accuracy' → the metric used to rank each combination
param_grid = {
    'max_depth':         [3, 5, 7, 10, None],  # controls tree depth (None = unlimited)
    'min_samples_split': [2, 5, 10],           # minimum samples required to split a node
    'min_samples_leaf':  [1, 2, 4],            # minimum samples required at each leaf
}
# Total: 5 × 3 × 3 = 45 combinations × 5 folds = 225 model fits

grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,              # 5-fold cross-validation per combination
    scoring='accuracy',
    n_jobs=-1,         # use all CPU cores in parallel
    verbose=0,         # 0=silent, 1=progress bar, 2=detailed
    return_train_score=True  # also record training scores (helps spot overfitting)
)
grid_search.fit(X_train_s, y_train)

print(f'\n=== GridSearchCV Results (Decision Tree) ===')
print(f'Total combinations tried : {len(grid_search.cv_results_["params"])}')
print(f'Best hyperparameters     : {grid_search.best_params_}')
print(f'Best cross-val accuracy  : {grid_search.best_score_:.4f}')

# best_estimator_ = the model already retrained on full X_train with best params
best_grid_model = grid_search.best_estimator_
grid_acc = accuracy_score(y_test, best_grid_model.predict(X_test_s))
print(f'Test accuracy            : {grid_acc:.4f}  (improvement: {(grid_acc-base_acc)*100:+.1f}%)')

# Show top 5 parameter combinations by cross-val score
print(f'\n=== Top 5 Hyperparameter Combinations ===')
results_df = pd.DataFrame(grid_search.cv_results_)
top5 = results_df[['params', 'mean_test_score', 'std_test_score']].sort_values(
    'mean_test_score', ascending=False
).head(5)
for _, row in top5.iterrows():
    print(f"  CV={row['mean_test_score']:.4f} ± {row['std_test_score']:.4f}  params={row['params']}")

# ── RANDOMIZED SEARCH CV ──────────────────────────────────────────────────────
# RandomizedSearchCV: samples n_iter RANDOM combinations instead of all
# Advantage: much faster when the parameter space is large
# Research shows: 60 random trials often find 90%+ of what exhaustive search finds
# Use randint(a, b) → random integer from a to b-1
param_dist = {
    'n_neighbors': randint(1, 30),        # random K from 1 to 29
    'weights':     ['uniform', 'distance'],  # uniform=all neighbors equal, distance=closer wins
    'p':           [1, 2],                # 1=Manhattan distance, 2=Euclidean distance
}

rand_search = RandomizedSearchCV(
    estimator=KNeighborsClassifier(),
    param_distributions=param_dist,
    n_iter=30,          # sample 30 random combinations (instead of all possible)
    cv=5,
    scoring='accuracy',
    random_state=42,    # reproducible random sampling
    n_jobs=-1
)
rand_search.fit(X_train_s, y_train)

rand_acc = accuracy_score(y_test, rand_search.best_estimator_.predict(X_test_s))
print(f'\n=== RandomizedSearchCV Results (KNN) ===')
print(f'Random combinations tried : 30  (fast vs exhaustive GridSearch)')
print(f'Best hyperparameters      : {rand_search.best_params_}')
print(f'Best cross-val accuracy   : {rand_search.best_score_:.4f}')
print(f'Test accuracy             : {rand_acc:.4f}')

# ── CROSS-VALIDATION (standalone) ────────────────────────────────────────────
# A single train/test split can be "lucky" or "unlucky" depending on which rows land where
# K-Fold CV trains+evaluates the model K times on different splits → more reliable average
# StratifiedKFold ensures each fold preserves the class ratio of the full dataset
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    best_grid_model, X_train_s, y_train,
    cv=cv_strategy,
    scoring='accuracy'
)

print(f'\n=== 5-Fold Stratified Cross-Validation (Best Decision Tree) ===')
for i, score in enumerate(cv_scores, 1):
    bar = '█' * int(score * 30)
    print(f'  Fold {i}: {score:.4f}  {bar}')
print(f'  Mean   : {cv_scores.mean():.4f}')
print(f'  Std Dev: {cv_scores.std():.4f}  (low std = stable across different data splits)')

# ── SUMMARY ───────────────────────────────────────────────────────────────────
print('\n=== Accuracy Comparison ===')
comparison = pd.DataFrame({
    'Method':       ['Baseline (defaults)', 'After GridSearchCV (DT)', 'After RandomizedSearch (KNN)'],
    'Accuracy':     [f'{base_acc:.4f}', f'{grid_acc:.4f}', f'{rand_acc:.4f}'],
    'Improvement':  ['+0.00%', f'{(grid_acc-base_acc)*100:+.1f}%', f'{(rand_acc-base_acc)*100:+.1f}%'],
})
print(comparison.to_string(index=False))
print(f'\nKey insight: GridSearchCV  = slow but thorough (best for small grids)')
print(f'             RandomizedSearchCV = fast but approximate (best for large grids)')

In [ ]:
## ── PIPELINE & COLUMNTRANSFORMER ─────────────────────────────────────────────

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

# Realistic dataset with mixed types: numeric columns + categorical columns + missing values
np.random.seed(42)
n = 300
df_raw = pd.DataFrame({
    'StudyHours': np.random.uniform(1, 10, n),
    'Attendance': np.random.uniform(40, 100, n),
    'PastScore':  np.random.uniform(30, 100, n),
    'SleepHours': np.random.uniform(4, 10, n),
    'Gender':     np.random.choice(['Male', 'Female'], n),                      # categorical
    'Internet':   np.random.choice(['Yes', 'No', None], n, p=[0.5, 0.4, 0.1]), # categorical with NaN
})
# Introduce missing values in numeric columns to simulate real-world data
df_raw.loc[np.random.choice(n, 20, replace=False), 'StudyHours'] = np.nan
df_raw.loc[np.random.choice(n, 15, replace=False), 'PastScore']  = np.nan

passed = (df_raw['StudyHours'].fillna(5) * 5 + df_raw['Attendance'] * 0.4 +
          np.random.normal(0, 10, n)) > 80
y = passed.astype(int)

X_train, X_test, y_train, y_test = train_test_split(df_raw, y, test_size=0.2, random_state=42)

print('=== Raw Training Data ===')
print(f'Shape: {X_train.shape}  |  Missing values per column:')
print(X_train.isnull().sum()[X_train.isnull().sum() > 0])

# ── COLUMNTRANSFORMER ─────────────────────────────────────────────────────────
# Different column types need different preprocessing
# ColumnTransformer lets you define a separate pipeline per column group,
# then automatically merges all results into one matrix

numeric_features     = ['StudyHours', 'Attendance', 'PastScore', 'SleepHours']
categorical_features = ['Gender', 'Internet']

# Sub-pipeline for numeric columns: impute missing values → scale
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),  # fill NaN with column mean
    ('scaler',  StandardScaler()),                 # standardise to mean=0, std=1
])

# Sub-pipeline for categorical columns: impute missing → one-hot encode
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),  # fill NaN with most common value
    ('encoder', OneHotEncoder(handle_unknown='ignore')),   # creates binary columns per category
    # handle_unknown='ignore' → if a new category appears at test time, encode it as all-zeros
])

# ColumnTransformer: applies each sub-pipeline to its designated columns
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline,     numeric_features),
    ('cat', categorical_pipeline, categorical_features),
])

# ── PIPELINE: Preprocessor + Model ───────────────────────────────────────────
# Pipeline chains multiple steps: output of one step → input of next step
# Key benefit: prevents DATA LEAKAGE
#   → Without pipeline: you might accidentally fit the scaler on all data before splitting
#   → With pipeline: fit() on X_train → scaler learns ONLY from training stats
#                    predict() on X_test → applies the SAME stats (no re-fitting)

full_pipeline = Pipeline([
    ('preprocessor', preprocessor),                                  # step 1: clean + encode + scale
    ('classifier',   DecisionTreeClassifier(max_depth=5, random_state=42)),  # step 2: model
])

# pipeline.fit() calls each step's fit_transform() in order
full_pipeline.fit(X_train, y_train)

# pipeline.predict() calls each step's transform() then the model's predict()
y_pred = full_pipeline.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f'\n=== Pipeline Results (no tuning) ===')
print(f'Accuracy: {acc:.4f}')
print(classification_report(y_test, y_pred, target_names=['Fail', 'Pass']))

# ── PIPELINE + GRIDSEARCHCV ────────────────────────────────────────────────────
# Pipeline works seamlessly with GridSearchCV
# Reference nested parameters with double underscore: stepname__paramname
# e.g. 'classifier__max_depth' tunes the DecisionTreeClassifier's max_depth
param_grid = {
    'classifier__max_depth':         [3, 5, 7, 10],
    'classifier__min_samples_split': [2, 5, 10],
    # You can also tune preprocessor parameters:
    # 'preprocessor__num__imputer__strategy': ['mean', 'median'],
}

grid = GridSearchCV(full_pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)

best_acc = accuracy_score(y_test, grid.best_estimator_.predict(X_test))
print(f'=== Pipeline + GridSearchCV ===')
print(f'Best params  : {grid.best_params_}')
print(f'Best CV score: {grid.best_score_:.4f}')
print(f'Test accuracy: {best_acc:.4f}  (improvement over baseline: {(best_acc-acc)*100:+.1f}%)')

print('\n=== Why Use a Pipeline? ===')
print('  ✓ Prevents data leakage — imputer/scaler fitted on training data only')
print('  ✓ Clean API — one .fit() and one .predict() handles everything')
print('  ✓ Reusable — save the pipeline, load it, call .predict() on new raw data')
print('  ✓ GridSearchCV compatible — tune model AND preprocessor params together')

In [ ]:
## ── MODEL SAVING & LOADING ───────────────────────────────────────────────────

import joblib
import pickle
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import make_classification
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Train a simple pipeline to demonstrate saving and loading
X, y = make_classification(n_samples=300, n_features=4, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  DecisionTreeClassifier(max_depth=5, random_state=42)),
])
pipeline.fit(X_train, y_train)

original_acc = accuracy_score(y_test, pipeline.predict(X_test))
original_preds = pipeline.predict(X_test)
print(f'Trained pipeline accuracy: {original_acc:.4f}')

# ── SAVING WITH JOBLIB ─────────────────────────────────────────────────────────
# joblib is the RECOMMENDED way to save sklearn models
# Faster than pickle for objects that contain large numpy arrays (which sklearn models do)
# Supports optional compression: joblib.dump(model, path, compress=3)

path_joblib = 'student_model.joblib'
joblib.dump(pipeline, path_joblib)  # serialise pipeline to disk

file_size = os.path.getsize(path_joblib)
print(f'\n=== Saved with joblib ===')
print(f'File: {path_joblib}  ({file_size} bytes on disk)')

# Reload — creates an exact copy of the original Python object
pipeline_jl = joblib.load(path_joblib)
jl_acc = accuracy_score(y_test, pipeline_jl.predict(X_test))
preds_match = np.array_equal(original_preds, pipeline_jl.predict(X_test))
print(f'Reloaded accuracy : {jl_acc:.4f}  (same as original: {jl_acc == original_acc})')
print(f'Predictions match : {preds_match}  (every prediction is identical)')

# ── SAVING WITH PICKLE ─────────────────────────────────────────────────────────
# pickle is Python's built-in serialization — works for ANY Python object
# Slightly slower than joblib for large arrays, but perfectly fine for small models
# 'wb' = write binary mode (required for pickle)
# 'rb' = read binary mode

path_pickle = 'student_model.pkl'
with open(path_pickle, 'wb') as f:
    pickle.dump(pipeline, f)

file_size_pkl = os.path.getsize(path_pickle)
print(f'\n=== Saved with pickle ===')
print(f'File: {path_pickle}  ({file_size_pkl} bytes on disk)')

with open(path_pickle, 'rb') as f:
    pipeline_pkl = pickle.load(f)

pkl_acc = accuracy_score(y_test, pipeline_pkl.predict(X_test))
print(f'Reloaded accuracy : {pkl_acc:.4f}')

# ── JOBLIB VS PICKLE ───────────────────────────────────────────────────────────
print('\n=== joblib vs pickle ===')
comparison = pd.DataFrame({
    'Feature':         ['Speed (large models)', 'Compression', 'sklearn support',  'Best for'],
    'joblib':          ['Faster',              'Built-in',    'Recommended',       'sklearn pipelines / large arrays'],
    'pickle':          ['Slower',              'None',        'Works fine',        'Small models / any Python object'],
})
print(comparison.to_string(index=False))

print('\nRecommendation: use joblib for sklearn models, pickle for general Python objects')

# ── REAL-WORLD PATTERN ────────────────────────────────────────────────────────
print('''
=== Real-World Usage Pattern ===

── TRAINING SCRIPT (train_model.py) ──────────────────────────────────────────
  pipeline = Pipeline([('scaler', StandardScaler()), ('model', DecisionTreeClassifier())])
  pipeline.fit(X_train, y_train)
  joblib.dump(pipeline, 'model_v1.joblib')          # save to disk
  print('Model saved')

── INFERENCE / DEPLOYMENT SCRIPT (predict.py) ────────────────────────────────
  pipeline = joblib.load('model_v1.joblib')          # load ONCE at startup
  new_data  = pd.read_csv('new_students.csv')
  preds     = pipeline.predict(new_data)             # preprocessing is automatic
  print(preds)

Key point: because the Pipeline includes the scaler, the loaded model handles
raw (unscaled) data automatically — no manual preprocessing step needed.
''')

# Cleanup demo files
os.remove(path_joblib)
os.remove(path_pickle)
print('Demo files cleaned up.')

# ── PHASE 6 COMPLETE SUMMARY ──────────────────────────────────────────────────
print('''
=== Phase 6 Summary ===

GridSearchCV:
  → Tries every combination in param_grid (exhaustive)
  → Best for small grids (< ~100 combinations)
  → grid_search.best_params_   → winning hyperparameters
  → grid_search.best_estimator_ → already-fitted best model

RandomizedSearchCV:
  → Samples n_iter random combinations (approximate but fast)
  → Best for large grids or continuous distributions
  → Works the same API as GridSearchCV

StratifiedKFold + cross_val_score:
  → K-fold CV preserves class ratio in each fold
  → More reliable accuracy estimate than a single train/test split

Pipeline:
  → Pipeline([('step1', transformer), ('step2', model)])
  → .fit(X_train) fits every step; .predict(X_test) applies every step
  → Prevents data leakage, works with GridSearchCV via step__param syntax

ColumnTransformer:
  → Apply different pipelines to different columns simultaneously
  → preprocessor = ColumnTransformer([('num', num_pipe, num_cols),
                                       ('cat', cat_pipe, cat_cols)])

Model Saving:
  → joblib.dump(model, 'file.joblib')   /  joblib.load('file.joblib')
  → pickle.dump(model, open('file.pkl','wb'))  /  pickle.load(open('file.pkl','rb'))
''')